# 🧬 Foldseek matching - Azure ML Job Submit
---
이 노트북은 `download_alphafold_structure.py`를 사용하여 AlphaFold PDB 구조를 대규모로 수집하는 작업을 제출합니다.
비동기(Async) 다운로드를 통해 속도를 최적화했습니다.

In [1]:
from pathlib import Path
from azure.ai.ml import MLClient, command, Input
from azure.ai.ml.entities import Environment
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml.constants import AssetTypes
import json
from datetime import datetime

print("✅ 패키지 import 완료")

✅ 패키지 import 완료


## 🎛️ 작업 설정

In [2]:
# ============================================================================
# 🎯 실험 설정
# ============================================================================
EXPERIMENT_NAME = "Foldseek_Structure_Search"
DISPLAY_NAME = f"Foldseek_Matching_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# ============================================================================
# 🖥️ 컴퓨팅 설정
# ============================================================================
CLUSTER_NAME = "cpu-32core-cluster"  # GPU 클러스터 (다운로드 작업에는 과분하지만 요청대로 사용)
INSTANCE_COUNT = 1

# ============================================================================
# 🐍 환경 설정
# ============================================================================
ENVIRONMENT_NAME = "foldseek-env"
USE_EXISTING_ENV = False  # conda.yaml 수정 반영을 위해 업데이트 필요

# ============================================================================
# 📂 입출력 설정
# ============================================================================
# 소스 폴더 (이 안에 fasta_accessions.txt와 스크립트가 있어야 함)
SRC_DIR = "src"


print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Cluster: {CLUSTER_NAME}")

Experiment: Foldseek_Structure_Search
Cluster: cpu-32core-cluster


## 🔐 Azure ML 연결

In [3]:
print("=" * 70)
print("🔐 Azure ML Client 연결 시작")
print("=" * 70)

try:
    try:
        credential = DefaultAzureCredential()
        credential.get_token("https://management.azure.com/.default")
    except Exception:
        print("🌐 브라우저 인증(Interactive)으로 전환합니다...")
        credential = InteractiveBrowserCredential()

    ml_client = MLClient.from_config(credential=credential)
    ws = ml_client.workspaces.get(ml_client.workspace_name)

    print(f"✅ 연결 성공!")
    print(f"   - Workspace: {ws.name}")
    print(f"   - Resource Group: {ws.resource_group}")
    print(f"   - Subscription: {ml_client.subscription_id}")

except Exception as e:
    print(f"❌ 연결 실패: {e}")

🔐 Azure ML Client 연결 시작


DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' pa

🌐 브라우저 인증(Interactive)으로 전환합니다...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


✅ 연결 성공!
   - Workspace: REDACTED_AZURE_RESOURCE_GROUP-ml
   - Resource Group: REDACTED_AZURE_RESOURCE_GROUP
   - Subscription: REDACTED_AZURE_SUBSCRIPTION_ID


## 🐍 환경 설정 (aiohttp, tqdm 포함)

In [4]:
env_name = ENVIRONMENT_NAME

if USE_EXISTING_ENV:
    try:
        env = ml_client.environments.get(name=env_name, label="latest")
        print(f"✅ 기존 환경 사용: {env_name} (version: {env.version})")
    except Exception as e:
        print(f"⚠️ 기존 환경 찾을 수 없음: {e}")
        USE_EXISTING_ENV = False

if not USE_EXISTING_ENV:
    conda_yaml_path = Path("env/conda.yaml")
    if not conda_yaml_path.exists():
        raise FileNotFoundError(f"env/conda.yaml 파일을 찾을 수 없습니다.")
    
    env = Environment(
        name=env_name,
        description="Foldseek Structure Search (CPU Optimized)",
        conda_file=str(conda_yaml_path),
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    )
    env = ml_client.environments.create_or_update(env)
    print(f"✅ 환경 생성/업데이트 완료: {env_name} (version: {env.version})")

✅ 환경 생성/업데이트 완료: foldseek-env (version: 2)


## 🚀 Job 생성 및 제출

# alphafold 3D 구조 파일 보고 Foldseek에 입력

In [ ]:
from azure.ai.ml import command, Input
# Output 임포트는 이제 필요 없지만, 놔둬도 상관없습니다.

# [데이터 경로] 
# (주의: 아까 Storage Explorer에서 확인한 zip 파일이 들어있는 '폴더' 자산 이름이어야 합니다)
DATA_PATH = "azureml:alphafold_structures_folder:3" 

job = command(
    code="./src",
    
    # [핵심 수정 1] output_dir을 ${{outputs.results}} 변수 대신
    # Azure가 무조건 파일을 수거해가는 고정 경로 './outputs'로 변경했습니다.
    command="python foldseek_driver.py --input_data ${{inputs.structures}} --output_dir ./outputs",
    
    inputs={
        "structures": Input(
            type="uri_folder", 
            path=DATA_PATH,
            mode="download"    # zip 파일 압축 해제 속도를 위해 download 모드 유지
        )
    },
    
    # [핵심 수정 2] outputs={...} 설정을 아예 삭제했습니다.
    # 이걸 삭제해야 Azure가 "./outputs" 폴더에 있는 파일을 자동으로 Artifact로 업로드합니다.
    # outputs={"results": Output(type="uri_folder")}, 
    
    environment=f"{ENVIRONMENT_NAME}@latest",
    compute=CLUSTER_NAME,
    timeout_minutes=600 
)

# 제출
print(f"🚀 Job 제출 중... (데이터: {DATA_PATH})")
ml_client.jobs.create_or_update(job)

Warnings: [timeout_minutes: Unknown field.]
Warnings: [timeout_minutes: Unknown field.]


🚀 Job 제출 중... (데이터: azureml:alphafold_structures_folder:3)


Experiment,Name,Type,Status,Details Page
SeekFoldazureml로돌리기,brave_tiger_d3t5s411qj,command,Starting,Link to Azure Machine Learning studio


In [17]:
# 아까 성공했다는 그 작업 이름 (사진 보고 적음)
JOB_NAME = "amusing_yam_7j17jf8vtr"

# Azure한테 "그 작업 정보 좀 내놔봐" 하고 묻는 코드
job = ml_client.jobs.get(JOB_NAME)

# "그래서 결과물은 어디 있는데?" 하고 주소 출력
print("=============== [찾았다! 요놈!] ===============")
print(f"Job 상태: {job.status}")
if "results" in job.outputs:
    saved_path = job.outputs["results"].path
    print(f"💎 파일이 숨어있는 진짜 주소:\n{saved_path}")
    print("==============================================")
else:
    print("😱 outputs['results'] 키가 안 보임! 대신 있는 것들:", job.outputs.keys())

=============== [찾았다! 요놈!] ===============
Job 상태: Completed
💎 파일이 숨어있는 진짜 주소:
None


# 다운받은거 sort 하기

In [1]:
import pandas as pd
import glob
import os

# ==========================================
# [설정] 다운로드 받은 폴더 경로를 여기에 넣기
# 예: r"C:\Users\User\Downloads\outputs"
input_folder = r"C:\Users\EL19\Downloads\outputs"
# ==========================================

def merge_local():
    # 파일 찾기
    print(f"📂 파일 찾는 중: {input_folder}")
    all_files = glob.glob(os.path.join(input_folder, "**", "results_batch_*.tsv"), recursive=True)
    
    if not all_files:
        print("❌ 파일을 못 찾겠어. 경로 다시 확인해봐.")
        return

    print(f"📊 총 {len(all_files)}개 파일 발견. 합치는 중...")

    # 데이터 읽어서 합치기
    # 컬럼명은 foldseek 기본값 기준 (query, target, evalue, bits, ...)
    # 만약 파일 안에 헤더가 없으면 names옵션 사용, 있으면 header=0 사용
    # (아까 코드에선 convertalis 포맷을 지정했으니 헤더가 없을 확률이 높음)
    col_names = ["query", "target", "evalue", "bits", "alntmscore", "qcov", "tcov"]
    
    df_list = []
    for f in all_files:
        try:
            # 파일이 비어있을 수도 있으니 체크
            if os.path.getsize(f) > 0:
                df = pd.read_csv(f, sep='\t', names=col_names, header=None)
                df_list.append(df)
        except Exception as e:
            print(f"⚠️ 읽기 실패 (무시): {f} - {e}")

    if not df_list:
        print("❌ 합칠 데이터가 없음.")
        return

    merged_df = pd.concat(df_list, ignore_index=True)
    
    # 정렬 (Query 이름순 -> Evalue 좋은 순)
    print("🔄 정렬 중...")
    merged_df.sort_values(by=['query', 'evalue'], ascending=[True, True], inplace=True)

    # 저장
    save_path = "final_submission.tsv"
    merged_df.to_csv(save_path, sep='\t', index=False, header=False)
    
    print(f"🎉 끝! 저장 완료: {os.path.abspath(save_path)}")
    print(f"총 행 개수: {len(merged_df)}")

if __name__ == "__main__":
    merge_local()

📂 파일 찾는 중: C:\Users\EL19\Downloads\outputs
📊 총 30개 파일 발견. 합치는 중...
🔄 정렬 중...
🎉 끝! 저장 완료: c:\Users\EL19\OneDrive - (주)엘릭서\바탕 화면\MS_2nd_Project\SeekFold azureml로 돌리기\final_submission.tsv
총 행 개수: 2488745


##### 결과
파일 찾는 중: C:\Users\EL19\Downloads\outputs  
총 30개 파일 발견. 합치는 중...  
정렬 중...  
끝! 저장 완료: c:\Users\EL19\OneDrive - (주)엘릭서\바탕 화면\MS_2nd_Project\SeekFold azureml로 돌리기\final_submission.tsv  
총 행 개수: 2488745

In [2]:
import pandas as pd
import os

def clean_and_save():
    # 파일 이름 설정 (네가 말한 파일명으로 맞췄어)
    input_filename = "final_submission.tsv" 
    output_filename = "foldseek_result.tsv"

    # 파일이 있는지 먼저 확인
    if not os.path.exists(input_filename):
        print(f"❌ 오류: '{input_filename}' 파일이 안 보여! 경로랑 이름을 확인해줘.")
        # 혹시 오타난 이름으로 저장되어 있을까 봐 체크
        if os.path.exists("final_suvmission.tsv"):
            print("💡 팁: 'final_suvmission.tsv' 파일은 있네. 이걸로 읽을게.")
            input_filename = "final_suvmission.tsv"
        else:
            return

    print(f"📂 '{input_filename}' 읽는 중...")

    # 1. 데이터 불러오기
    # 아까 헤더 없이 저장했으니까, 컬럼 이름을 지정해서 읽어야 해.
    col_names = ["query", "target", "evalue", "bits", "alntmscore", "qcov", "tcov"]
    df = pd.read_csv(input_filename, sep='\t', names=col_names)

    # 2. ID 정제 작업 (핵심!)
    # '_'를 기준으로 1번만 자르고, 그 뒷부분([-1])을 가져옴
    # 예: '000023_P12345' -> 'P12345'
    # 만약 '_'가 없으면 원래 문자열 그대로 유지됨 (안전함)
    df['query'] = df['query'].apply(lambda x: str(x).split('_', 1)[-1])

    # 3. 결과 저장
    # index=False: 행 번호 빼기
    # header=False: 데이터만 깔끔하게 저장 (CAFA 제출용 포맷 유지)
    df.to_csv(output_filename, sep='\t', index=False, header=False)

    print("-" * 30)
    print(f"🎉 작업 완료! '{output_filename}' 저장됨.")
    print("-" * 30)
    print("▼ 변환된 데이터 미리보기 (앞부분 숫자가 사라졌는지 확인해봐)")
    print(df.head())

if __name__ == "__main__":
    clean_and_save()

📂 'final_submission.tsv' 읽는 중...
------------------------------
🎉 작업 완료! 'foldseek_result.tsv' 저장됨.
------------------------------
▼ 변환된 데이터 미리보기 (앞부분 숫자가 사라졌는지 확인해봐)
        query      target        evalue  bits  alntmscore   qcov   tcov
0  A0A8F4S8P5  A0A8F4S8P5  4.578000e-22  1167      1.0060  1.000  1.000
1  A0A8F4S8P5  A0A7L9EYL3  1.989000e-16   865      0.9919  1.000  1.000
2  A0A8F4S8P5  A0A8F4NWB9  2.093000e-14   755      0.9813  1.000  1.000
3  A0A8F4S8P5  A0A1W5T1Y3  2.032000e-06   323      0.8737  0.994  0.983
4  A0A8F4S8P5      S8B3I8  3.252000e-06   315      0.8745  0.994  0.983


##### 결과 
작업 완료! 'foldseek_result.tsv' 저장됨.

In [3]:
import pandas as pd
import os

def extract_missing_ids():
    accession_file = "fasta_accessions.txt"
    result_file = "foldseek_result.tsv"
    output_file = "no_alphafold_structure.txt"

    # 파일 있는지 체크
    if not os.path.exists(accession_file) or not os.path.exists(result_file):
        print("❌ 파일이 없어! 경로 확인해 봐.")
        return

    print("🔍 분석 시작...")

    # 1. 전체 ID 로드 (Set으로 변환해서 중복 제거 및 검색 속도 향상)
    with open(accession_file, "r") as f:
        all_ids = set(line.strip() for line in f if line.strip())

    # 2. Foldseek 결과에서 Query ID만 로드
    # 0번째 컬럼이 Query ID임
    try:
        df = pd.read_csv(result_file, sep='\t', header=None, usecols=[0])
        found_ids = set(df[0].astype(str)) # 문자열로 확실하게 변환
    except Exception as e:
        print(f"⚠️ 결과 파일 읽다가 에러남: {e}")
        return

    # 3. 차집합 연산 (전체 - 찾은 것 = 없는 것)
    missing_ids = all_ids - found_ids

    # 4. 저장
    with open(output_file, "w") as f:
        for pid in sorted(missing_ids): # 보기 좋게 정렬해서 저장
            f.write(pid + "\n")

    print("-" * 30)
    print(f"📊 전체 ID 개수: {len(all_ids):,}개")
    print(f"✅ 구조 있는 ID: {len(found_ids):,}개")
    print(f"❌ 구조 없는 ID: {len(missing_ids):,}개")
    print("-" * 30)
    print(f"🎉 저장 완료: {output_file}")

if __name__ == "__main__":
    extract_missing_ids()

🔍 분석 시작...
------------------------------
📊 전체 ID 개수: 224,309개
✅ 구조 있는 ID: 145,521개
❌ 구조 없는 ID: 78,788개
------------------------------
🎉 저장 완료: no_alphafold_structure.txt


##### 결과
전체 ID 개수: 224,309개  
구조 있는 ID: 145,521개  
구조 없는 ID: 78,788개   
🎉 저장 완료: no_alphafold_structure.txt

# 점수 정규화 하는 파일 만들기

In [1]:
import pandas as pd
from collections import defaultdict
from pathlib import Path

FOLDSEEK_HITS = "foldseek_result.tsv"
TRAIN_TERMS   = "validation_superset_term.tsv"
OUT_FILE      = "structure_go_scores(최신화).tsv"

hits = pd.read_csv(
    FOLDSEEK_HITS,
    sep="\t",
    header=None,
    names=["query", "target", "evalue", "bits", "alntmscore", "qcov", "tcov"]
)

# self-hit 제거
hits = hits[hits["query"] != hits["target"]].copy()

# 가중치: alntmscore (1.0 초과는 1.0으로 고정)
hits["w"] = hits["alntmscore"].clip(upper=1.0)

terms = pd.read_csv(TRAIN_TERMS, sep="\t")  # protein_id, go_id, aspect

# 정답지(Train Data) 메모리에 올리기
prot2go = defaultdict(list)
for r in terms.itertuples(index=False):
    prot2go[r.EntryID].append((r.term, r.aspect))

# 점수 계산 로직 
# 이는 점수 합산 방식임
# 
records = []
for query, g in hits.groupby("query", sort=False):
    go_scores = defaultdict(float)
    for r in g.itertuples(index=False):
        if r.target not in prot2go:
            continue
        for go_id, aspect in prot2go[r.target]:
            go_scores[(go_id, aspect)] += r.w

    for (go_id, aspect), score in go_scores.items():
        records.append((query, go_id, aspect, score))

out = pd.DataFrame(records, columns=["protein_id", "go_id", "aspect", "score"])
out.to_csv(OUT_FILE, sep="\t", index=False)

print("완료:", OUT_FILE)
print("query 수:", out["protein_id"].nunique())
print("총 row:", len(out))


완료: structure_go_scores(최신화).tsv
query 수: 70557
총 row: 644529


##### 결과 
완료: structure_go_scores(최신화).tsv
query 수: 70557
총 row: 644529 

구조 있는 ID: 145,521개 -> query 수: 70557개 로 줄은 이유는 **자기자신 제외 구조 유사한 것이 하나도 안나온 것**

##### 데이터 흐름 요약
Foldseek 결과 (145,521개): "구조적으로 비슷한 놈은 다 찾았어요!"

Filter 1 (Self-hit): "자기 자신이랑 매칭된 건 뺍니다." -> (일부 탈락)

Filter 2 (No GO Terms): "매칭된 놈(Target)이 뭐 하는 놈인지 정답지에 안 써있으면 뺍니다." -> (대거 탈락 📉)

최종 결과 (70557개): "구조도 비슷하고, 그 놈의 기능 정보도 확실한 것만 남았습니다."

# 정규화 및 필터링

In [2]:
import pandas as pd

IN = "structure_go_scores(최신화).tsv"

df = pd.read_csv(IN, sep="\t")
# 1) sanity (데이터 무셜성 확인)
print("proteins:", df["protein_id"].nunique())
per_q = df.groupby("protein_id").size()
print("GO per protein: min/med/mean/max =",
      per_q.min(), per_q.median(), round(per_q.mean(),2), per_q.max())
print("aspect counts:\n", df["aspect"].value_counts())
print("score summary:\n", df["score"].describe(percentiles=[0.5, 0.95]))

# 2) max-normalize per protein (최대값 정규화) ⭐핵심
df["score_norm"] = df["score"] / df.groupby("protein_id")["score"].transform("max")

# 3) try a setting (ex: Top50 + >=0.2) Filtering (Top-k & Threshold)
df_sorted = df.sort_values(["protein_id", "score_norm"], ascending=[True, False])
df_top = df_sorted.groupby("protein_id").head(50)
df_filt = df_top[df_top["score_norm"] >= 0.2].copy()

print("\nAFTER filter:")
print("proteins:", df_filt["protein_id"].nunique())
per_q2 = df_filt.groupby("protein_id").size()
print("GO per protein: min/med/mean/max =",
      per_q2.min(), per_q2.median(), round(per_q2.mean(),2), per_q2.max())

# 4) aspect split output (final score=score_norm for now) (파일 분리)
for asp, name in [("F","pred_mf_test.tsv"),("P","pred_bp_test.tsv"),("C","pred_cc_test.tsv")]:
    out = df_filt[df_filt["aspect"]==asp][["protein_id","go_id","score_norm"]]
    out.to_csv(name, sep="\t", index=False, header=False)
    print("wrote", name, "rows:", len(out))


proteins: 70557
GO per protein: min/med/mean/max = 1 5.0 9.13 177
aspect counts:
 aspect
P    332771
C    178973
F    132785
Name: count, dtype: int64
score summary:
 count    644529.000000
mean          0.713002
std           0.358276
min           0.050420
50%           0.723700
95%           1.000000
max           8.999400
Name: score, dtype: float64

AFTER filter:
proteins: 70557
GO per protein: min/med/mean/max = 1 5.0 8.8 50
wrote pred_mf_test.tsv rows: 130879
wrote pred_bp_test.tsv rows: 313065
wrote pred_cc_test.tsv rows: 176985


# 📊 Foldseek GO 예측 결과 요약

#### 1. 필터링 전 (Raw Data)
- **총 단백질:** 127,438개
- **GO 수/단백질:** 평균 37.8개 (최대 478개)
- **점수:** 합산 방식 (최대 15.67) → 정규화 필요

#### 2. 필터링 후 (Top-50 & Cutoff 0.2)
- **총 단백질:** 127,438개 (100% 생존 ✅)
- **GO 수/단백질:** 평균 17.2개 (최대 50개로 제한됨 ✅)
- **점수:** 0.0 ~ 1.0 사이로 정규화 완료

#### 3. 최종 생성 파일 (Propagation용)
| 파일명 | Aspect | 행 개수 (Rows) |
| :--- | :---: | :---: |
| `pred_mf_test.tsv` | 🧪 MF | 528,115 |
| `pred_bp_test.tsv` | 🧬 BP | 980,746 |
| `pred_cc_test.tsv` | 🏢 CC | 679,584 |
| **Total** | | **2,188,445** |

# 상향전파
##### MAX 전파 (Max Propagation) 방식
어떤 부모 노드(Parent)의 점수는 **"자기 자신의 점수"**와 "자기 자식들(Children)의 점수" 중 **가장 큰 값(Max)**을 따라간다.  

자식(이중 가닥 복구) 점수가 0.9면 $\rightarrow$ 부모(DNA 복구)도 최소 0.9는 되어야 함 $\rightarrow$ 조상(대사 과정)도 0.9는 되어야 함.

이렇게 점수를 위로 쭉쭉 올려주면(Propagation), 맞출 수 있는 정답의 개수(Recall)가 확 늘어나서 F1-score가 떡상해.

In [5]:
import os
from azure.ai.ml import command, Input, Output
from azure.ai.ml import MLClient
from azure.identity import InteractiveBrowserCredential
from azure.ai.ml.constants import AssetTypes

# 1. 로그인 (팝업 창 뜨면 로그인!)
print("🔐 로그인 시도 중... 브라우저를 확인해!")
credential = InteractiveBrowserCredential()
ml_client = MLClient.from_config(credential=credential)

# 2. 데이터 폴더 (여기에 tsv, pkl, npz 다 있어야 함)
LOCAL_DATA_FOLDER = "./data_for_propagation" 

# 3. Job 설정
job = command(
    code="./src",  # 소스 코드 폴더
    command="python propagate_parallel.py --input_dir ${{inputs.input_files}} --output_dir ${{outputs.final_result}}",
    
    inputs={
        "input_files": Input(
            type=AssetTypes.URI_FOLDER,
            path=LOCAL_DATA_FOLDER,
            mode="download" # 컴퓨트로 다운로드
        )
    },
    
    outputs={
        "final_result": Output(
            type=AssetTypes.URI_FOLDER,
            # [핵심] upload 모드여야 작업 후 Blob Storage에 파일이 남음
            mode="upload" 
        )
    },
    
    environment="foldseek-env@latest", 
    compute="FOR-CAFA-6", 
    display_name="Parallel_Prop_32Core_Final"
)

# 4. 제출
print(f"🚀 Job 제출 중... (32코어 풀가동 + 결과 영구 저장)")
returned_job = ml_client.jobs.create_or_update(job)
print(f"✅ Job Created! URL: {returned_job.studio_url}")
print("👉 작업이 끝나면 Studio의 'Outputs + logs' 탭 -> 'Data outputs' -> 'final_result'를 확인해!")

Found the config file in: .\config.json
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed


Overriding of current MeterProvider is not allowed


🔐 로그인 시도 중... 브라우저를 확인해!


Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


🚀 Job 제출 중... (32코어 풀가동 + 결과 영구 저장)


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading data_for_propagation (26.02 

✅ Job Created! URL: https://ml.azure.com/runs/lime_pipe_37ggf34jjy?wsid=/subscriptions/REDACTED_AZURE_SUBSCRIPTION_ID/resourcegroups/REDACTED_AZURE_RESOURCE_GROUP/workspaces/REDACTED_AZURE_RESOURCE_GROUP-ml&tid=5fb256f0-fbf2-40d2-81d5-bac1b32c419d
👉 작업이 끝나면 Studio의 'Outputs + logs' 탭 -> 'Data outputs' -> 'final_result'를 확인해!


In [6]:
import os
from azure.ai.ml import command, Input, Output, MLClient
from azure.identity import InteractiveBrowserCredential
from azure.ai.ml.constants import AssetTypes

# 1. 로그인
print("🔐 로그인 창이 뜨면 로그인해주세요.")
credential = InteractiveBrowserCredential()
ml_client = MLClient.from_config(credential=credential)

# 2. 입력 데이터 폴더 지정
# 아까 다운로드 받은, 'propagated.tsv' 3개가 들어있는 폴더 경로를 적어!
# 예: "./final_result" 또는 "./data_for_propagation" (파일이 어디 있느냐에 따라 다름)
LOCAL_DATA_FOLDER = "./final_result" 

# 3. Job 설정
job = command(
    code="./src", 
    command="python make_submission.py --input_dir ${{inputs.input_files}} --output_dir ${{outputs.final_result}}",
    
    inputs={
        "input_files": Input(
            type=AssetTypes.URI_FOLDER,
            path=LOCAL_DATA_FOLDER, 
            mode="download"
        )
    },
    
    outputs={
        "final_result": Output(
            type=AssetTypes.URI_FOLDER,
            mode="upload" # 결과 영구 저장
        )
    },
    
    environment="foldseek-env@latest", 
    compute="FOR-CAFA-6", 
    display_name="Make_Submission_Top500"
)

# 4. 제출
print(f"🚀 Submission 생성 Job 제출 중...")
returned_job = ml_client.jobs.create_or_update(job)
print(f"✅ Job Created! URL: {returned_job.studio_url}")

Found the config file in: .\config.json
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


🔐 로그인 창이 뜨면 로그인해주세요.
🚀 Submission 생성 Job 제출 중...


Uploading src (0.02 MBs): 100%|##########| 19259/19259 [00:00<00:00, 38023.82it/s]


Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy 'C:\Users\EL19\OneDrive - (주)엘릭서\바탕 화면\MS_2nd_Project\SeekFold azureml로 돌리기\final_result' 'https://8ai2ndteam3ml0864856365.blob.core.windows.net/azureml-blobstore-66f388de-e162-420f-9e94-0a104f34bccb/LocalUpload/e65270737a59a7acdb637762894152ac3859cee8e4cee88f9525ecd5f2fff920/final_result' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.
Uploading final_result (535.69 MBs): 100%|##########| 535688363/535688363 [00:46<00:00, 11446496.65it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


✅ Job Created! URL: https://ml.azure.com/runs/teal_animal_7r83ggx4vc?wsid=/subscriptions/REDACTED_AZURE_SUBSCRIPTION_ID/resourcegroups/REDACTED_AZURE_RESOURCE_GROUP/workspaces/REDACTED_AZURE_RESOURCE_GROUP-ml&tid=5fb256f0-fbf2-40d2-81d5-bac1b32c419d
